In [1]:
!pip install pycryptodome

DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [2]:
import os
from Crypto.Cipher import ChaCha20

In [3]:
# --- Stream cipher helpers ---
def encrypt_message(key, plaintext, nonce=None):
    """
    Encrypt a message with ChaCha20.
    Returns (ciphertext, nonce).
    If no nonce is given, a random 8-byte nonce is generated.
    """
    if nonce is None:
        nonce = os.urandom(8)
    cipher = ChaCha20.new(key=key, nonce=nonce)
    ciphertext = cipher.encrypt(plaintext)
    return ciphertext, nonce

def decrypt_message(key, ciphertext, nonce):
    """
    Decrypt a message with ChaCha20 given key and nonce.
    """
    cipher = ChaCha20.new(key=key, nonce=nonce)
    return cipher.decrypt(ciphertext)


# --- Simulated chat ---
print("=== Alice ↔ Bob Secure Chat Simulation ===")

# Shared symmetric key (e.g. derived via Diffie-Hellman beforehand)
shared_key = os.urandom(32)

# Conversation (sequence of messages)
conversation = [
    ("Alice", b"Hi Bob, how are you today?"),
    ("Bob",   b"Hey Alice, I am fine, thanks! What about you?"),
    ("Alice", b"I'm great. Want to meet later?"),
    ("Bob",   b"Sure, let's meet at 6 PM."),
]

# --- Messaging loop ---
transcript = []

for sender, msg in conversation:
    # Encrypt with a fresh nonce
    ciphertext, nonce = encrypt_message(shared_key, msg)
    
    # Store what was "sent" (header + ciphertext)
    transcript.append((sender, msg, ciphertext, nonce))
    
    # Show what goes over the network
    print(f"\n[{sender}] sends:")
    print("  Nonce (header):", nonce.hex())
    print("  Ciphertext:    ", ciphertext.hex())
    
    # Receiver decrypts
    decrypted = decrypt_message(shared_key, ciphertext, nonce)
    print("  Receiver decrypts ->", decrypted.decode())




=== Alice ↔ Bob Secure Chat Simulation ===

[Alice] sends:
  Nonce (header): cefe78632b465061
  Ciphertext:     d7f01573d165e9a597e55ea9d4d5ed2587afe13ea4a47837aac7
  Receiver decrypts -> Hi Bob, how are you today?

[Bob] sends:
  Nonce (header): bbdb4c70eedcb61d
  Ciphertext:     dbcd0f0b74219738490a975ffbd8c5bc161b3e1d996d9126f77de92e083a165e57084eb43e8be3e6b9b5115507
  Receiver decrypts -> Hey Alice, I am fine, thanks! What about you?

[Alice] sends:
  Nonce (header): d155a8b043b51713
  Ciphertext:     492ff4e70dafb59f49a658b11ead6871a38320940843cc562d141e9bddd0
  Receiver decrypts -> I'm great. Want to meet later?

[Bob] sends:
  Nonce (header): 81a8124c948f10c1
  Ciphertext:     130ea6dac9d551d34fc7e5183249230be0476eba4c5cfa8958
  Receiver decrypts -> Sure, let's meet at 6 PM.


In [4]:
# --- Security failure: nonce reuse demo ---
print("\n=== Nonce reuse attack (two-time pad) ===")

m1 = b"Hello, this is Alice"
m2 = b"I need help from you"

# Bad practice: same nonce used for both
bad_nonce = os.urandom(8)

c1, _ = encrypt_message(shared_key, m1, nonce=bad_nonce)
c2, _ = encrypt_message(shared_key, m2, nonce=bad_nonce)

# Attacker computes XOR of ciphertexts
xor_ct = bytes([a ^ b for a, b in zip(c1, c2)])
xor_pt = bytes([a ^ b for a, b in zip(m1, m2)])

print("C1 XOR C2:", xor_ct)
print("M1 XOR M2:", xor_pt)

print("\n>>> Notice: C1 XOR C2 = M1 XOR M2. Reusing the same nonce leaks information!")


=== Nonce reuse attack (two-time pad) ===
C1 XOR C2: b'\x01E\x02\t\nH\x00\x1c\r\x05\x03\x00\x0f\x01O,L\x10\x0c\x10'
M1 XOR M2: b'\x01E\x02\t\nH\x00\x1c\r\x05\x03\x00\x0f\x01O,L\x10\x0c\x10'

>>> Notice: C1 XOR C2 = M1 XOR M2. Reusing the same nonce leaks information!
